# Chest X-ray — FULL run (all 7 models)

**Run only AFTER colab_small.ipynb returned GO.** All 7 models, full epochs, live progress. Same entrypoint. Cells 1-6 identical to the small run.

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone / update code from GitHub

In [ ]:
import os
REPO = 'https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    !git clone $REPO /content/Chest_Xray
else:
    !git -C /content/Chest_Xray pull
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — unzip data to fast local disk (slow, once per session)
Live unzip progress shown per file.

In [ ]:
import os, glob
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('unzipping', os.path.basename(z), flush=True)
    !unzip -q -o "$z" -d data/raw
print('folders:', os.listdir('data/raw'))

## Cell 4 — bring the manifest

In [ ]:
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE_DIR}/manifest.csv'):
    shutil.copy(f'{DRIVE_DIR}/manifest.csv','data/manifest.csv'); print('manifest copied')
else:
    !python scripts/build_manifest.py

## Cell 5 — install deps + GPU check (GPU must NOT be empty)
pip progress streams live.

In [ ]:
!pip -q install -r requirements.txt
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 6 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df = pd.read_csv('data/manifest.csv')
missing = [p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
if missing:
    print('EXAMPLE:', missing[0]); print('folders:', os.listdir('data/raw'))
    print('>>> STOP: fix folder mapping before running.')
else:
    print('>>> OK: paths resolve. Continue.')

## Cell 7 — FULL pipeline on all 7 models (LIVE progress; hours on T4)
`--resume` skips already-trained models, so safe to re-run after a disconnect.
⚠️ If results/lenet5 holds an old 1-epoch smoke checkpoint, delete it first (next cell).

In [ ]:
MODELS = 'densenet201 efficientnetb0 resnet50 mobilenetv3large xception vit lenet5'
!python -u scripts/run_pipeline.py --models $MODELS --epochs 100 --batch-size 64

## Cell 8 — final C3 coupling (all 7 models)

In [ ]:
import json
print(json.dumps(json.load(open('results/c3_coupling.json')), indent=2))

## Cell 9 — list final outputs (figures + tables)

In [ ]:
import os
print('FIGURES:', os.listdir('results/figures'))
print('TABLES :', os.listdir('results/tables'))
print('STATS  :', os.path.exists('results/stats_summary.json'))

## Cell 10 — SAVE everything to Drive (do not skip)

In [ ]:
import datetime
stamp = datetime.date.today().isoformat()
!cp -r results /content/drive/MyDrive/cxr_data/results_full_$stamp
print('saved -> Drive/cxr_data/results_full_'+stamp)